In [1]:
# Colab cell 1
!rm -rf /content/chr-op-alns
!git clone https://github.com/ahmetbengoz/chr-op-alns.git /content/chr-op-alns

%cd /content/chr-op-alns

# requirements boş ya da eksik olabilir diye temel paketleri de ayrıca kuruyorum
!pip install -q pandas numpy matplotlib openpyxl xlsxwriter
!pip install -q -r requirements.txt || true

import os
print("Repo root:", os.getcwd())
print("code exists:", os.path.exists("code/main_pipeline.py"))
print("data exists:", os.path.exists("data"))
print("dataset exists:", os.path.exists("data/dataset_instances.xlsx"))

Cloning into '/content/chr-op-alns'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 66 (delta 24), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 388.95 KiB | 13.89 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/chr-op-alns
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.3 MB/s eta 0:00:00
Repo root: /content/chr-op-alns
code exists: True
data exists: True
dataset exists: True


In [3]:
# Colab cell 2
from pathlib import Path
import re

src_path = Path("/content/chr-op-alns/code/main_pipeline.py")
src = src_path.read_text(encoding="utf-8")

print("Toplam karakter:", len(src))
print("0.02 geçen yer sayısı:", src.count("0.02"))

# 0.02 geçen bağlamları göster
matches = list(re.finditer(r"0\.02", src))
for i, m in enumerate(matches[:10], 1):
    start = max(0, m.start() - 120)
    end = min(len(src), m.end() + 120)
    print(f"\n--- Match {i} ---")
    print(src[start:end])

Toplam karakter: 24305
0.02 geçen yer sayısı: 1

--- Match 1 ---
S acceptance/objective.
# Keep as-is if you want EXACT behavior consistent with the latest pipeline runs.
LAMBDA_WAIT = 0.02

# -----------------------------
# Output folders (repo-friendly)
# -----------------------------
OUT_DATA = "data"
OUT


In [4]:
# Colab cell 3
from pathlib import Path
import re

src_path = Path("/content/chr-op-alns/code/main_pipeline.py")
original_src = src_path.read_text(encoding="utf-8")

def make_patched_script(lambda_value: float, out_path: str):
    src = original_src
    replaced = False

    # 1) En olası açık tanımlar
    patterns = [
        r'(?m)^(\s*LAMBDA\s*=\s*)0\.02(\s*)$',
        r'(?m)^(\s*LAMBDA_W\s*=\s*)0\.02(\s*)$',
        r'(?m)^(\s*LAMBDA_WAIT\s*=\s*)0\.02(\s*)$',
        r'(?m)^(\s*LAMBDA_WEIGHT\s*=\s*)0\.02(\s*)$',
        r'(?m)^(\s*OBJ_LAMBDA\s*=\s*)0\.02(\s*)$',
        r'(?m)^(\s*WAIT_PENALTY\s*=\s*)0\.02(\s*)$',
    ]

    for pat in patterns:
        new_src, n = re.subn(pat, rf'\g<1>{lambda_value}\2', src)
        if n > 0:
            src = new_src
            replaced = True
            print(f"[OK] Pattern matched: {pat}")
            break

    # 2) Yukarıdakiler yoksa bağlamlı replace:
    # scalarization / objective / lambda yorumuna yakın ilk 0.02'yi değiştir
    if not replaced:
        context_patterns = [
            r'((?:lambda|scalariz|objective|waiting|penalt)[^\\n]{0,120}?)(0\.02)',
        ]
        for pat in context_patterns:
            def repl(m):
                return m.group(1) + str(lambda_value)
            new_src, n = re.subn(pat, repl, src, count=1, flags=re.IGNORECASE)
            if n > 0:
                src = new_src
                replaced = True
                print(f"[OK] Context pattern matched: {pat}")
                break

    # 3) Hâlâ olmadıysa güvenli şekilde dur
    if not replaced:
        raise RuntimeError(
            "λ satırı otomatik bulunamadı. Cell 2 çıktısını bana atarsan net patch satırını çıkarırım."
        )

    Path(out_path).write_text(src, encoding="utf-8")
    print(f"[OK] Patched script written to: {out_path}")

# test patch
make_patched_script(0.02, "/content/chr-op-alns/code/main_pipeline_lambda_test.py")

[OK] Pattern matched: (?m)^(\s*LAMBDA_WAIT\s*=\s*)0\.02(\s*)$
[OK] Patched script written to: /content/chr-op-alns/code/main_pipeline_lambda_test.py


In [5]:
# Colab cell 4
import subprocess
import sys
from pathlib import Path

test_script = "/content/chr-op-alns/code/main_pipeline_lambda_test.py"

result = subprocess.run(
    [sys.executable, test_script],
    cwd="/content/chr-op-alns",
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT (son 4000 karakter):\n")
print(result.stdout[-4000:])
print("\nSTDERR (son 4000 karakter):\n")
print(result.stderr[-4000:])

RETURN CODE: 0

STDOUT (son 4000 karakter):

DONE.
Outputs:
- data/dataset_instances.xlsx
- data/results_table.xlsx
- data/Table1_method_comparison.xlsx
- data/Table2_fleet_size.xlsx
- figures/figure2_convergence.png
- figures/figure3_fleet_size.png
- logs/run_log.txt


STDERR (son 4000 karakter):




In [6]:
# Colab cell 5
import os
import shutil
import subprocess
import sys
from pathlib import Path

repo_root = Path("/content/chr-op-alns")
runs_root = repo_root / "runs_lambda_sensitivity"
runs_root.mkdir(exist_ok=True)

lambda_values = [0, 0.01, 0.02, 0.05, 0.1]

summary = []

for lam in lambda_values:
    print("\n" + "="*80)
    print(f"RUNNING LAMBDA = {lam}")
    print("="*80)

    lam_tag = str(lam).replace(".", "p")
    patched_script = repo_root / "code" / f"main_pipeline_lambda_{lam_tag}.py"
    out_dir = runs_root / f"lambda_{lam_tag}"
    out_dir.mkdir(exist_ok=True)

    make_patched_script(lam, str(patched_script))

    result = subprocess.run(
        [sys.executable, str(patched_script)],
        cwd=str(repo_root),
        capture_output=True,
        text=True
    )

    # stdout/stderr log
    (out_dir / "stdout.txt").write_text(result.stdout, encoding="utf-8")
    (out_dir / "stderr.txt").write_text(result.stderr, encoding="utf-8")

    # oluşan klasörleri kopyala
    for folder in ["data", "figures", "logs"]:
        src_folder = repo_root / folder
        if src_folder.exists():
            dst_folder = out_dir / folder
            if dst_folder.exists():
                shutil.rmtree(dst_folder)
            shutil.copytree(src_folder, dst_folder)

    summary.append({
        "lambda": lam,
        "returncode": result.returncode,
        "out_dir": str(out_dir)
    })

    print("RETURN CODE:", result.returncode)
    if result.returncode != 0:
        print("HATA. stderr son kısmı:")
        print(result.stderr[-2000:])
        break

print("\nSUMMARY")
for row in summary:
    print(row)


RUNNING LAMBDA = 0
[OK] Pattern matched: (?m)^(\s*LAMBDA_WAIT\s*=\s*)0\.02(\s*)$
[OK] Patched script written to: /content/chr-op-alns/code/main_pipeline_lambda_0.py
RETURN CODE: 0

RUNNING LAMBDA = 0.01
[OK] Pattern matched: (?m)^(\s*LAMBDA_WAIT\s*=\s*)0\.02(\s*)$
[OK] Patched script written to: /content/chr-op-alns/code/main_pipeline_lambda_0p01.py
RETURN CODE: 0

RUNNING LAMBDA = 0.02
[OK] Pattern matched: (?m)^(\s*LAMBDA_WAIT\s*=\s*)0\.02(\s*)$
[OK] Patched script written to: /content/chr-op-alns/code/main_pipeline_lambda_0p02.py
RETURN CODE: 0

RUNNING LAMBDA = 0.05
[OK] Pattern matched: (?m)^(\s*LAMBDA_WAIT\s*=\s*)0\.02(\s*)$
[OK] Patched script written to: /content/chr-op-alns/code/main_pipeline_lambda_0p05.py
RETURN CODE: 0

RUNNING LAMBDA = 0.1
[OK] Pattern matched: (?m)^(\s*LAMBDA_WAIT\s*=\s*)0\.02(\s*)$
[OK] Patched script written to: /content/chr-op-alns/code/main_pipeline_lambda_0p1.py
RETURN CODE: 0

SUMMARY
{'lambda': 0, 'returncode': 0, 'out_dir': '/content/chr-op-alns/

In [7]:
# Cell 6A
from pathlib import Path

runs_root = Path("/content/chr-op-alns/runs_lambda_sensitivity")

for d in sorted(runs_root.iterdir()):
    if d.is_dir():
        print("\n###", d.name)
        data_dir = d / "data"
        if data_dir.exists():
            for x in sorted(data_dir.glob("*.xlsx")):
                print(" -", x.name)


### lambda_0
 - Table1_method_comparison.xlsx
 - Table2_fleet_size.xlsx
 - dataset_instances.xlsx
 - results_table.xlsx

### lambda_0p01
 - Table1_method_comparison.xlsx
 - Table2_fleet_size.xlsx
 - dataset_instances.xlsx
 - results_table.xlsx

### lambda_0p02
 - Table1_method_comparison.xlsx
 - Table2_fleet_size.xlsx
 - dataset_instances.xlsx
 - results_table.xlsx

### lambda_0p05
 - Table1_method_comparison.xlsx
 - Table2_fleet_size.xlsx
 - dataset_instances.xlsx
 - results_table.xlsx

### lambda_0p1
 - Table1_method_comparison.xlsx
 - Table2_fleet_size.xlsx
 - dataset_instances.xlsx
 - results_table.xlsx


In [8]:
# Cell 7A
import pandas as pd
from pathlib import Path

runs_root = Path("/content/chr-op-alns/runs_lambda_sensitivity")

for d in sorted(runs_root.iterdir()):
    if d.is_dir():
        path = d / "data" / "results_table.xlsx"
        if path.exists():
            print("\n" + "="*80)
            print(d.name)
            print("="*80)
            xl = pd.ExcelFile(path)
            print("SHEETS:", xl.sheet_names)
            for sh in xl.sheet_names:
                df = pd.read_excel(path, sheet_name=sh)
                print(f"\nSHEET: {sh} | shape={df.shape}")
                print("COLUMNS:", list(df.columns))
                print(df.head(3))
                break


lambda_0
SHEETS: ['Sheet1']

SHEET: Sheet1 | shape=(4344, 12)
COLUMNS: ['instance_id', 'A', 'n_items', 'density', 'cap', 'pickers', 'robots', 'method', 'makespan', 'total_wait', 'robot_wait', 'cpu_s']
  instance_id   A  n_items density  cap  pickers  robots        method  \
0       I0001  10      200     low    1        2       1    Integrated   
1       I0001  10      200     low    1        2       1  NoCongestion   
2       I0001  10      200     low    1        2       1        NoSync   

   makespan  total_wait  robot_wait  cpu_s  
0       648          12          12    0.0  
1       632          10          10    0.0  
2       540           0           0    0.0  

lambda_0p01
SHEETS: ['Sheet1']

SHEET: Sheet1 | shape=(4344, 12)
COLUMNS: ['instance_id', 'A', 'n_items', 'density', 'cap', 'pickers', 'robots', 'method', 'makespan', 'total_wait', 'robot_wait', 'cpu_s']
  instance_id   A  n_items density  cap  pickers  robots        method  \
0       I0001  10      200     low    1   

In [9]:
# Cell 8A
from pathlib import Path

runs_root = Path("/content/chr-op-alns/runs_lambda_sensitivity")

for d in sorted(runs_root.iterdir()):
    if d.is_dir():
        print("\n" + "="*80)
        print(d.name)
        print("="*80)

        for p in sorted(d.rglob("*")):
            if p.is_file() and p.suffix.lower() in [".xlsx", ".csv", ".txt", ".png"]:
                print(p.relative_to(d))


lambda_0
data/Table1_method_comparison.xlsx
data/Table2_fleet_size.xlsx
data/dataset_instances.xlsx
data/results_table.xlsx
figures/figure2_convergence.png
figures/figure3_fleet_size.png
logs/run_log.txt
stderr.txt
stdout.txt

lambda_0p01
data/Table1_method_comparison.xlsx
data/Table2_fleet_size.xlsx
data/dataset_instances.xlsx
data/results_table.xlsx
figures/figure2_convergence.png
figures/figure3_fleet_size.png
logs/run_log.txt
stderr.txt
stdout.txt

lambda_0p02
data/Table1_method_comparison.xlsx
data/Table2_fleet_size.xlsx
data/dataset_instances.xlsx
data/results_table.xlsx
figures/figure2_convergence.png
figures/figure3_fleet_size.png
logs/run_log.txt
stderr.txt
stdout.txt

lambda_0p05
data/Table1_method_comparison.xlsx
data/Table2_fleet_size.xlsx
data/dataset_instances.xlsx
data/results_table.xlsx
figures/figure2_convergence.png
figures/figure3_fleet_size.png
logs/run_log.txt
stderr.txt
stdout.txt

lambda_0p1
data/Table1_method_comparison.xlsx
data/Table2_fleet_size.xlsx
data/dat

In [10]:
# Cell 8B
import pandas as pd
from pathlib import Path

runs_root = Path("/content/chr-op-alns/runs_lambda_sensitivity")

for d in sorted(runs_root.iterdir()):
    if d.is_dir():
        print("\n" + "="*80)
        print(d.name)
        print("="*80)

        for xlsx in sorted((d / "data").glob("*.xlsx")):
            try:
                xl = pd.ExcelFile(xlsx)
                print(f"\nFILE: {xlsx.name}")
                print("SHEETS:", xl.sheet_names)
                for sh in xl.sheet_names:
                    df = pd.read_excel(xlsx, sheet_name=sh)
                    cols = list(df.columns)
                    if any(c in cols for c in ["makespan", "total_wait", "robot_wait", "cpu_s"]):
                        print(f"  SHEET: {sh} | shape={df.shape}")
                        print("  COLS :", cols)
                        print(df.head(3))
            except Exception as e:
                print("ERR:", xlsx.name, e)


lambda_0

FILE: Table1_method_comparison.xlsx
SHEETS: ['Sheet1']

FILE: Table2_fleet_size.xlsx
SHEETS: ['Integrated_All', 'ALNS_Subset']

FILE: dataset_instances.xlsx
SHEETS: ['Sheet1']

FILE: results_table.xlsx
SHEETS: ['Sheet1']
  SHEET: Sheet1 | shape=(4344, 12)
  COLS : ['instance_id', 'A', 'n_items', 'density', 'cap', 'pickers', 'robots', 'method', 'makespan', 'total_wait', 'robot_wait', 'cpu_s']
  instance_id   A  n_items density  cap  pickers  robots        method  \
0       I0001  10      200     low    1        2       1    Integrated   
1       I0001  10      200     low    1        2       1  NoCongestion   
2       I0001  10      200     low    1        2       1        NoSync   

   makespan  total_wait  robot_wait  cpu_s  
0       648          12          12    0.0  
1       632          10          10    0.0  
2       540           0           0    0.0  

lambda_0p01

FILE: Table1_method_comparison.xlsx
SHEETS: ['Sheet1']

FILE: Table2_fleet_size.xlsx
SHEETS: ['Integrat

In [11]:
import pandas as pd
from pathlib import Path

runs_root = Path("/content/chr-op-alns/runs_lambda_sensitivity")

rows = []

for d in sorted(runs_root.iterdir()):
    if d.is_dir():
        lam = d.name.replace("lambda_", "").replace("p",".")
        path = d / "data" / "results_table.xlsx"

        df = pd.read_excel(path)

        df_int = df[df["method"]=="Integrated"]

        rows.append({
            "lambda": float(lam),
            "mean_makespan": df_int["makespan"].mean(),
            "mean_total_wait": df_int["total_wait"].mean(),
            "mean_robot_wait": df_int["robot_wait"].mean()
        })

summary = pd.DataFrame(rows).sort_values("lambda")

summary

,lambda,mean_makespan,mean_total_wait,mean_robot_wait
0,0.00,906.473148,2025.156481,2025.156481
1,0.01,906.473148,2025.156481,2025.156481
2,0.02,906.473148,2025.156481,2025.156481
3,0.05,906.473148,2025.156481,2025.156481
4,0.10,906.473148,2025.156481,2025.156481


In [12]:
summary.to_csv("/content/lambda_summary.csv", index=False)